### Download and Load Dataset

In [9]:
import os
import kagglehub
import numpy as np
import pandas as pd

path = kagglehub.competition_download('playground-series-s6e9')

train_df = pd.read_csv(os.path.join(path, 'train.csv'))
test_df = pd.read_csv(os.path.join(path, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(path, 'sample_submission.csv'))

### EDA Summary

In [ ]:
cat_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
            'Subsidy_Available', 'Range_Anxiety_Level']

numeric_cols = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
                'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
                'Environmental_Concern_Level']

print(train_df['Will_Buy_EV'].value_counts(normalize=True))

Will_Buy_EV
No     0.825355
Yes    0.174645
Name: proportion, dtype: float64


- Target col imbalance.
- nominal: Gender, Current_Car_Type, 
- mapping(yes/no): Home_Charging_Possible, Subsidy_Available 
- ordinal: City_Type, Range_Anxiety_Level 
- 'Subsidy_Available' and 'Range_Anxiety_Level' are strongest pred

### Preprocess

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

nominal_cols = ['Gender', 'Current_Car_Type']
binary_cols = ['Home_Charging_Possible', 'Subsidy_Available']
ordinal_cols = ['City_Type', 'Range_Anxiety_Level']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('nom', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
    ('bin', OrdinalEncoder(categories=[['No', 'Yes'], ['No', 'Yes']]), binary_cols),
    ('ord', OrdinalEncoder(categories=[['Rural', 'Suburban', 'Urban'], ['Low', 'Medium', 'High']]), ordinal_cols),
])

X = train_df.drop(columns=['id', 'Will_Buy_EV'])
y = train_df['Will_Buy_EV'].map({'No': 0, 'Yes': 1})
X_test = test_df.drop(columns=['id'])

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

In [ ]:
def add_target_encoding(X_fit, y_fit, X_apply):
    out = X_apply.copy()

    combo_fit = X_fit['Subsidy_Available'] + "_" + X_fit['Range_Anxiety_Level']
    combo_apply = X_apply['Subsidy_Available'] + "_" + X_apply['Range_Anxiety_Level']
    combo_map = y_fit.groupby(combo_fit).mean()

    out['Combo_te'] = combo_apply.map(combo_map)

    return out

In [10]:
def run_cv(model, X, y, X_test, preprocessor, te_cols=['Combo_te'], n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_aucs = []

    full_preprocessor = ColumnTransformer(
        transformers=preprocessor.transformers + [('te', 'passthrough', te_cols)]
    )

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr_raw, X_va_raw = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        X_tr = add_target_encoding(X_tr_raw, y_tr, X_tr_raw)
        X_va = add_target_encoding(X_tr_raw, y_tr, X_va_raw)
        X_te = add_target_encoding(X_tr_raw, y_tr, X_test)

        pipe = Pipeline(steps=[('preprocessor', full_preprocessor), ('model', model)])
        pipe.fit(X_tr, y_tr)

        fold_pred = pipe.predict_proba(X_va)[:, 1]
        oof_preds[val_idx] = fold_pred
        test_preds += pipe.predict_proba(X_te)[:, 1] / n_splits

        fold_auc = roc_auc_score(y_va, fold_pred)
        fold_aucs.append(fold_auc)
        print(f"Fold {fold}: AUC = {fold_auc:.5f}")

    oof_auc = roc_auc_score(y, oof_preds)
    print(f"\nMean fold AUC: {np.mean(fold_aucs):.5f}")
    print(f"Pooled OOF AUC: {oof_auc:.5f}")
    return oof_preds, test_preds, oof_auc

In [11]:
from lightgbm import LGBMClassifier

lgbm_oof, lgbm_test_pred, lgbm_auc = run_cv(
    LGBMClassifier(random_state=42, scale_pos_weight=scale_pos_weight, max_bin=511,
                    n_estimators=500, learning_rate=0.05, num_leaves=63,
                    max_depth=5, min_child_samples=50, verbose=-1),
    X, y, X_test, preprocessor
)

Fold 1: AUC = 0.94104
Fold 2: AUC = 0.94181
Fold 3: AUC = 0.94313
Fold 4: AUC = 0.94281
Fold 5: AUC = 0.94233

Mean fold AUC: 0.94223
Pooled OOF AUC: 0.94222


In [13]:
submission = sample_sub.copy()
submission['Will_Buy_EV'] = lgbm_test_pred
submission.to_csv('submission/submission_final.csv', index=False)
print(submission.head())


       id  Will_Buy_EV
0  668665     0.058338
1  668666     0.081671
2  668667     0.021456
3  668668     0.016669
4  668669     0.081601
